# BbValidator: как это работает

Живой проход пайплайна: PDB → биофизический фронтенд → графовый энкодер → многозадачные головы → MC-Dropout вердикт.

Нужны обученный чекпоинт `checkpoints/best_model.pth` и образцы в `data/`. Соответствует модулям [02](../docs/02-biofizicheskiy-frontend.md) и [03](../docs/03-model-i-obuchenie.md) документации.

In [1]:
# Jupyter открывает тетрадь в notebooks/, а пути проекта относительны от корня.
# Поднимаемся до корня репозитория при необходимости.
import os
while not os.path.exists("inference.py") and os.getcwd() != os.path.dirname(os.getcwd()):
    os.chdir("..")
assert os.path.exists("inference.py"), "Откройте тетрадь внутри репозитория BbValidator"
print("Корень репозитория:", os.getcwd())

Корень репозитория: /home/pc/PycharmProjects/BbValidator


In [2]:
import glob
import torch

from inference import _autocast_ctx, build_model, center_coords, parse_pdb_to_backbone
from model.heads_loss import predict_with_uncertainty
from filter_designability import FAILURE_MODE_NAMES

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Устройство:", device)
model = build_model("checkpoints/best_model.pth", device)

/home/pc/PycharmProjects/BbValidator/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Устройство: cuda
Загрузка весов из checkpoints/best_model.pth...


## 1. Парсинг backbone

Из PDB берутся только атомы остова: `[N, Cα, C]` на остаток → тензор `[B, N, 3, 3]`, координаты центрируются. Стереохимия восстанавливается из геометрии остова — ни последовательность, ни сайдчейны не нужны.

In [3]:
coords_np = center_coords(parse_pdb_to_backbone("data/2RJV.pdb"))
coords = torch.from_numpy(coords_np).unsqueeze(0).to(device)
mask = torch.ones((1, coords.shape[1]), dtype=torch.bool, device=device)

print(f"Длина: {coords.shape[1]} остатков | тензор: {tuple(coords.shape)} = (B, N, 3 атома, 3 координаты)")
coords[0, :2]  # первые два остатка: N, Cα, C

Длина: 67 остатков | тензор: (1, 67, 3, 3) = (B, N, 3 атома, 3 координаты)


tensor([[[ 7.1931, 17.3472, -2.0727],
         [ 5.8340, 17.2802, -2.6657],
         [ 4.7270, 17.2362, -1.6137]],

        [[ 3.5030, 17.5342, -2.0397],
         [ 2.3420, 17.5222, -1.1497],
         [ 2.2541, 16.1582, -0.4687]]], device='cuda:0')

## 2. Биофизический фронтенд

Детерминированная «физика» (считается один раз):
- **геометрия** — торсионы φ/ψ/ω, CA-дистанции, счётчики H-связей и стерических конфликтов;
- **прокси дизайнуемости** — плотность упаковки, экспонированность растворителю, PCA-проекции фрагментов;
- **граф** — kNN по Cα (k=16) с RBF-кодированием дистанций.

На выходе узловые признаки `[B, N, 31]` = геометрия (10) + прокси (21).

In [4]:
with torch.no_grad():
    geom = model.frontend.geometry(coords, mask)
    front = model.frontend(coords, mask)

print("Сырые геометрические диагностики (всего по структуре):")
print(f"  стерические конфликты (Cβ < 3.5 Å, |i-j| ≥ 3): {int(geom['clash_count'].sum() / 2)}")
print(f"  H-связи остова: {int(geom['hbond_count'].sum())}")
print()
print("Узловые признаки:", tuple(front["node_feats"].shape))
print("Рёбра графа:", front["edge_indices"][0].shape[1], "(k=16 соседей × N остатков)")
front["node_feats"][0, :3, :8].float().round(decimals=3)

Сырые геометрические диагностики (всего по структуре):
  стерические конфликты (Cβ < 3.5 Å, |i-j| ≥ 3): 1
  H-связи остова: 51

Узловые признаки: (1, 67, 31)
Рёбра графа: 1072 (k=16 соседей × N остатков)


tensor([[ 0.0000,  0.0000,  0.2990, -0.9540,  0.0000,  0.0000,  0.4070,  0.0000],
        [-0.8150,  0.5790,  0.6650, -0.7470,  0.0100, -1.0000,  0.4010,  0.0000],
        [-0.8650,  0.5010,  0.6990, -0.7150,  0.0110, -1.0000,  0.4040,  0.0000]],
       device='cuda:0')

## 3. Энкодер + головы + MC-Dropout

Фронтенд прогоняется один раз, затем `mc_runs` проходов энкодера с включённым дропаутом: разброс предсказаний — эпистемическая неопределённость. Головы: P(fold), предсказанный RMSD, стерика, failure mode.

In [5]:
with torch.no_grad(), _autocast_ctx(device):
    mc = predict_with_uncertainty(model, coords, mask, mc_runs=16)
    preds = model(coords, mask)

pfold = mc["p_foldable"].item()
unc = mc["uncertainty"].item()
rmsd = max(0.0, torch.expm1(preds["rmsd"]).item())
p_steric = torch.sigmoid(preds["steric"]).item()
fm = torch.softmax(preds["failure_mode"].float(), dim=-1)[0]

print(f"P(fold)              = {pfold:.3f}  (MC-дисперсия {unc:.5f})")
print(f"Предсказанный RMSD   = {rmsd:.2f} Å")
print(f"Вероятность стерики  = {p_steric:.3f}")
print("Failure mode         =", {FAILURE_MODE_NAMES[i]: f"{v:.2f}" for i, v in enumerate(fm.tolist())})

P(fold)              = 0.816  (MC-дисперсия 0.00073)
Предсказанный RMSD   = 0.23 Å
Вероятность стерики  = 0.508
Failure mode         = {'ok': '0.74', 'easy': '0.00', 'hard': '0.00', 'near_native': '0.00', 'borderline': '0.26', 'unknown': '0.00'}


## 4. Все образцы в data/

Скоринг каждого файла по отдельности (B=1) и ранжирование по P(fold).

In [6]:
import pandas as pd

rows = []
for path in sorted(glob.glob("data/*.pdb")):
    c = torch.from_numpy(center_coords(parse_pdb_to_backbone(path))).unsqueeze(0).to(device)
    m = torch.ones((1, c.shape[1]), dtype=torch.bool, device=device)
    with torch.no_grad(), _autocast_ctx(device):
        r = predict_with_uncertainty(model, c, m, mc_runs=16)
        p = model(c, m)
        clashes = int((model.frontend.geometry(c, m)["clash_count"].sum() / 2).item())
    rows.append({
        "структура": path.split("/")[-1],
        "L": c.shape[1],
        "pfold": round(r["p_foldable"].item(), 3),
        "uncertainty": round(r["uncertainty"].item(), 5),
        "rmsd_pred": round(max(0.0, torch.expm1(p["rmsd"]).item()), 2),
        "конфликты": clashes,
    })

pd.DataFrame(rows).sort_values("pfold", ascending=False).reset_index(drop=True)

,структура,L,pfold,uncertainty,rmsd_pred,конфликты
0,my_protein_5.pdb,216,0.957,0.00002,0.02,0
1,my_protein_1.pdb,204,0.953,0.00004,0.02,1
2,3MYC.pdb,64,0.949,0.00004,0.06,0
3,my_protein_4.pdb,216,0.945,0.00004,0.02,0
4,my_protein_3.pdb,202,0.922,0.00008,0.02,0
5,2RJV.pdb,67,0.832,0.00050,0.23,1
6,my_protein_0.pdb,234,0.832,0.00056,0.04,1
7,my_protein_2.pdb,242,0.001,0.00000,3.59,2
8,2RJW.pdb,134,0.000,0.00000,19.75,1


Этот же пайплайн в батчевом виде использует `filter_designability.py` — фильтр по дизайнуемости с настраиваемыми гейтами (P(fold), стерика, конфликты, RMSD, неопределённость) и подробным вердиктом, какие метрики завалены:

```bash
python filter_designability.py -i data/ood/evodiff/scaffolds --min-pfold 0.5 --max-clashes 2
```